# ConsultAI: AI Opportunity Prioritisation Engine

**Decision science, Monte Carlo simulation, constrained portfolio selection and an executive planning application**

**Author:** Jorgo Luka  
**Format:** Advanced portfolio laboratory / standalone Google Colab application  
**Domain:** AI consulting and investment governance  
**Decision owner:** Chief Digital Officer and transformation steering committee

> This is an educational portfolio application. Its generated or synthetic results are not production, clinical, lending, investment or commercial claims. Run every cell in order and use only outputs you personally verify.

## Learning outcomes

- Translate an ambiguous transformation question into a transparent decision model.
- Quantify downside and uncertainty with reproducible Monte Carlo simulation.
- Compare greedy and exact constrained portfolio-selection strategies.
- Communicate recommendations through tests, audit artefacts and an interactive app.


## Project brief

**Decision question:** Which AI initiatives should an organisation fund, defer or reject when value, delivery uncertainty, readiness, risk and budget all matter?

You are acting as the analyst and application engineer. Your submission must move through six assessed stages:

1. Frame the decision and define measurable success.
2. Build or load data and enforce a data contract.
3. Explore patterns without contaminating evaluation data.
4. Develop the analytical or machine-learning solution.
5. Evaluate uncertainty, limitations, operational risks and business trade-offs.
6. Package the result as a tested application for a non-technical decision owner.

### Assessment rubric

| Dimension | Weight | Evidence expected |
|---|---:|---|
| Problem framing | 10% | Decision, user, scope and success criteria |
| Data engineering | 20% | Reproducibility, contracts, validation and lineage |
| Analytical depth | 25% | Appropriate methods, baselines and diagnostics |
| Evaluation | 20% | Metrics, uncertainty, trade-offs and failure analysis |
| Application engineering | 15% | Working interface, validation and audit trail |
| Communication | 10% | Clear recommendation, limitations and next actions |


## 0. Environment and reproducibility

Install the small UI dependency, define deterministic configuration and load the shared engineering toolkit.


In [1]:
!pip -q install "gradio>=5,<7"


In [2]:
PROJECT_SLUG = 'consultai'
PROJECT_TITLE = 'ConsultAI: AI Opportunity Prioritisation Engine'


In [3]:
from __future__ import annotations

import contextlib
import dataclasses
import datetime as dt
import hashlib
import io
import json
import math
import os
import platform
import random
import statistics
import sys
import time
import traceback
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")


@dataclass(frozen=True)
class ProjectConfig:
    slug: str
    title: str
    seed: int = 42
    currency: str = "GBP"
    confidence_level: float = 0.95
    launch_app: bool = False
    save_artifacts: bool = True
    sample_mode: bool = False

    def validate(self) -> None:
        if not self.slug or not self.slug.replace("_", "").isalnum():
            raise ValueError("slug must contain letters, digits or underscores")
        if not 0.50 < self.confidence_level < 1.0:
            raise ValueError("confidence_level must be between 0.50 and 1.0")
        if self.seed < 0:
            raise ValueError("seed must be non-negative")


CONFIG = ProjectConfig(
    slug=PROJECT_SLUG,
    title=PROJECT_TITLE,
    seed=42,
    launch_app=False,
    save_artifacts=True,
    sample_mode=False,
)
CONFIG.validate()


def in_colab() -> bool:
    return "google.colab" in sys.modules


def runtime_root() -> Path:
    return Path("/content") if in_colab() else Path.cwd()


ARTIFACT_ROOT = runtime_root() / f"{CONFIG.slug}_application_artifacts"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


def set_global_seed(seed: int) -> np.random.Generator:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    return np.random.default_rng(seed)


RNG = set_global_seed(CONFIG.seed)


def utc_now() -> str:
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def to_native(value: Any) -> Any:
    if dataclasses.is_dataclass(value):
        return {key: to_native(item) for key, item in asdict(value).items()}
    if isinstance(value, Mapping):
        return {str(key): to_native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_native(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (pd.Timestamp, dt.datetime, dt.date)):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value) if np.isscalar(value) else False:
        return None
    return value


def stable_hash(payload: Any, length: int = 16) -> str:
    serialised = json.dumps(to_native(payload), sort_keys=True, default=str)
    return hashlib.sha256(serialised.encode("utf-8")).hexdigest()[:length]


def frame_fingerprint(frame: pd.DataFrame, length: int = 16) -> str:
    ordered = frame.sort_index(axis=1).copy()
    hashed = pd.util.hash_pandas_object(ordered, index=True).values.tobytes()
    return hashlib.sha256(hashed).hexdigest()[:length]


def memory_megabytes(frame: pd.DataFrame) -> float:
    return float(frame.memory_usage(index=True, deep=True).sum() / 1_000_000)


def write_json(payload: Any, filename: str) -> Path:
    target = ARTIFACT_ROOT / filename
    target.write_text(json.dumps(to_native(payload), indent=2, sort_keys=True), encoding="utf-8")
    return target


def write_csv(frame: pd.DataFrame, filename: str, index: bool = False) -> Path:
    target = ARTIFACT_ROOT / filename
    frame.to_csv(target, index=index)
    return target


def save_figure(figure: plt.Figure, filename: str) -> Path:
    target = ARTIFACT_ROOT / filename
    figure.savefig(target, dpi=160, bbox_inches="tight")
    return target


@contextlib.contextmanager
def timed_step(name: str):
    started = time.perf_counter()
    print(f"START {name}")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - started
        print(f"DONE  {name} in {elapsed:,.2f}s")


def capture_versions(packages: Sequence[Any]) -> dict[str, str]:
    versions = {"python": platform.python_version(), "platform": platform.platform()}
    for package in packages:
        name = getattr(package, "__name__", package.__class__.__name__)
        versions[name] = str(getattr(package, "__version__", "unknown"))
    return versions


@dataclass
class GateResult:
    name: str
    passed: bool
    observed: Any
    expectation: str
    severity: str = "error"


class QualitySuite:
    def __init__(self, suite_name: str):
        self.suite_name = suite_name
        self.results: list[GateResult] = []

    def add(
        self,
        name: str,
        condition: bool,
        observed: Any,
        expectation: str,
        severity: str = "error",
    ) -> GateResult:
        if severity not in {"error", "warning"}:
            raise ValueError("severity must be error or warning")
        result = GateResult(name, bool(condition), observed, expectation, severity)
        self.results.append(result)
        return result

    def require_columns(self, frame: pd.DataFrame, columns: Sequence[str]) -> GateResult:
        missing = sorted(set(columns) - set(frame.columns))
        return self.add(
            "required_columns",
            not missing,
            missing,
            f"all required columns present: {list(columns)}",
        )

    def require_unique(self, frame: pd.DataFrame, columns: Sequence[str]) -> GateResult:
        duplicate_rows = int(frame.duplicated(list(columns)).sum())
        return self.add(
            f"unique_{'_'.join(columns)}",
            duplicate_rows == 0,
            duplicate_rows,
            "zero duplicate keys",
        )

    def require_nonempty(self, frame: pd.DataFrame) -> GateResult:
        return self.add("nonempty", len(frame) > 0, len(frame), "at least one row")

    def require_no_infinite(self, frame: pd.DataFrame) -> GateResult:
        numeric = frame.select_dtypes(include=np.number)
        infinite = int(np.isinf(numeric.to_numpy()).sum()) if not numeric.empty else 0
        return self.add("no_infinite_values", infinite == 0, infinite, "zero infinite values")

    def require_range(self, series: pd.Series, lower: float, upper: float, name: str) -> GateResult:
        valid = series.dropna().between(lower, upper)
        invalid = int((~valid).sum())
        return self.add(name, invalid == 0, invalid, f"values between {lower} and {upper}")

    def require_allowed(self, series: pd.Series, allowed: Sequence[Any], name: str) -> GateResult:
        invalid_values = sorted(set(series.dropna()) - set(allowed))
        return self.add(name, not invalid_values, invalid_values, f"values in {list(allowed)}")

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame([to_native(result) for result in self.results])

    def summary(self) -> dict[str, Any]:
        errors = [result for result in self.results if not result.passed and result.severity == "error"]
        warnings_found = [result for result in self.results if not result.passed and result.severity == "warning"]
        return {
            "suite": self.suite_name,
            "checks": len(self.results),
            "passed": sum(result.passed for result in self.results),
            "failed_errors": len(errors),
            "failed_warnings": len(warnings_found),
            "healthy": not errors,
        }

    def assert_all(self) -> None:
        errors = [result for result in self.results if not result.passed and result.severity == "error"]
        if errors:
            details = "; ".join(f"{item.name}: {item.observed}" for item in errors)
            raise AssertionError(f"Quality suite {self.suite_name} failed: {details}")


def dataset_overview(frame: pd.DataFrame, name: str) -> dict[str, Any]:
    return {
        "name": name,
        "rows": int(len(frame)),
        "columns": int(frame.shape[1]),
        "memory_mb": round(memory_megabytes(frame), 3),
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
        "fingerprint": frame_fingerprint(frame),
    }


def missingness_table(frame: pd.DataFrame) -> pd.DataFrame:
    result = pd.DataFrame(
        {
            "column": frame.columns,
            "dtype": frame.dtypes.astype(str).values,
            "missing_count": frame.isna().sum().values,
            "missing_pct": (100.0 * frame.isna().mean()).round(3).values,
            "unique_count": frame.nunique(dropna=True).values,
        }
    )
    return result.sort_values(["missing_pct", "unique_count"], ascending=[False, False]).reset_index(drop=True)


def numeric_profile(frame: pd.DataFrame) -> pd.DataFrame:
    numeric = frame.select_dtypes(include=np.number)
    if numeric.empty:
        return pd.DataFrame(columns=["column", "mean", "std", "min", "median", "max", "skew"])
    rows = []
    for column in numeric.columns:
        series = numeric[column].dropna()
        rows.append(
            {
                "column": column,
                "mean": series.mean(),
                "std": series.std(),
                "min": series.min(),
                "median": series.median(),
                "max": series.max(),
                "skew": series.skew(),
            }
        )
    return pd.DataFrame(rows)


def categorical_profile(frame: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    categorical = frame.select_dtypes(exclude=np.number)
    rows = []
    for column in categorical.columns:
        counts = categorical[column].astype("string").value_counts(dropna=False).head(top_n)
        rows.append(
            {
                "column": column,
                "unique_count": int(categorical[column].nunique(dropna=True)),
                "top_values": counts.to_dict(),
            }
        )
    return pd.DataFrame(rows)


def bootstrap_statistic(
    values: Sequence[float],
    statistic: Callable[[np.ndarray], float] = np.mean,
    confidence: float = 0.95,
    repetitions: int = 2_000,
    seed: int = 42,
) -> dict[str, float]:
    array = np.asarray(values, dtype=float)
    array = array[np.isfinite(array)]
    if len(array) < 2:
        raise ValueError("bootstrap requires at least two finite values")
    generator = np.random.default_rng(seed)
    estimates = np.empty(repetitions, dtype=float)
    for index in range(repetitions):
        sample = generator.choice(array, size=len(array), replace=True)
        estimates[index] = statistic(sample)
    alpha = (1.0 - confidence) / 2.0
    lower, upper = np.quantile(estimates, [alpha, 1.0 - alpha])
    return {
        "estimate": float(statistic(array)),
        "lower": float(lower),
        "upper": float(upper),
        "confidence": confidence,
        "repetitions": repetitions,
    }


def bootstrap_difference(
    left: Sequence[float],
    right: Sequence[float],
    repetitions: int = 2_000,
    seed: int = 42,
) -> dict[str, float]:
    a = np.asarray(left, dtype=float)
    b = np.asarray(right, dtype=float)
    generator = np.random.default_rng(seed)
    estimates = np.empty(repetitions, dtype=float)
    for index in range(repetitions):
        a_sample = generator.choice(a, size=len(a), replace=True)
        b_sample = generator.choice(b, size=len(b), replace=True)
        estimates[index] = a_sample.mean() - b_sample.mean()
    lower, upper = np.quantile(estimates, [0.025, 0.975])
    return {
        "difference": float(a.mean() - b.mean()),
        "lower": float(lower),
        "upper": float(upper),
        "probability_positive": float((estimates > 0).mean()),
    }


def wilson_interval(successes: int, trials: int, confidence_z: float = 1.96) -> tuple[float, float]:
    if trials <= 0:
        return (float("nan"), float("nan"))
    proportion = successes / trials
    denominator = 1.0 + confidence_z**2 / trials
    centre = proportion + confidence_z**2 / (2.0 * trials)
    radius = confidence_z * math.sqrt(
        proportion * (1.0 - proportion) / trials + confidence_z**2 / (4.0 * trials**2)
    )
    return ((centre - radius) / denominator, (centre + radius) / denominator)


def population_stability_index(
    reference: Sequence[float],
    current: Sequence[float],
    bins: int = 10,
    epsilon: float = 1e-6,
) -> float:
    reference_array = np.asarray(reference, dtype=float)
    current_array = np.asarray(current, dtype=float)
    reference_array = reference_array[np.isfinite(reference_array)]
    current_array = current_array[np.isfinite(current_array)]
    edges = np.unique(np.quantile(reference_array, np.linspace(0.0, 1.0, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0] = -np.inf
    edges[-1] = np.inf
    reference_counts, _ = np.histogram(reference_array, bins=edges)
    current_counts, _ = np.histogram(current_array, bins=edges)
    reference_share = np.clip(reference_counts / reference_counts.sum(), epsilon, None)
    current_share = np.clip(current_counts / current_counts.sum(), epsilon, None)
    return float(np.sum((current_share - reference_share) * np.log(current_share / reference_share)))


def numeric_drift_report(
    reference: pd.DataFrame,
    current: pd.DataFrame,
    columns: Sequence[str],
) -> pd.DataFrame:
    rows = []
    for column in columns:
        psi = population_stability_index(reference[column], current[column])
        rows.append(
            {
                "feature": column,
                "reference_mean": reference[column].mean(),
                "current_mean": current[column].mean(),
                "mean_shift_std": (current[column].mean() - reference[column].mean())
                / max(reference[column].std(), 1e-9),
                "psi": psi,
                "status": "review" if psi >= 0.20 else "watch" if psi >= 0.10 else "stable",
            }
        )
    return pd.DataFrame(rows).sort_values("psi", ascending=False).reset_index(drop=True)


def classification_metrics(
    y_true: Sequence[int],
    probability: Sequence[float],
    threshold: float = 0.50,
) -> dict[str, float]:
    truth = np.asarray(y_true, dtype=int)
    probability_array = np.asarray(probability, dtype=float)
    prediction = probability_array >= threshold
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(truth, prediction)),
        "precision": float(precision_score(truth, prediction, zero_division=0)),
        "recall": float(recall_score(truth, prediction, zero_division=0)),
        "f1": float(f1_score(truth, prediction, zero_division=0)),
        "roc_auc": float(roc_auc_score(truth, probability_array)),
        "average_precision": float(average_precision_score(truth, probability_array)),
        "brier": float(brier_score_loss(truth, probability_array)),
    }


def regression_metrics(y_true: Sequence[float], prediction: Sequence[float]) -> dict[str, float]:
    truth = np.asarray(y_true, dtype=float)
    predicted = np.asarray(prediction, dtype=float)
    residual = truth - predicted
    denominator = np.where(np.abs(truth) < 1e-9, np.nan, np.abs(truth))
    return {
        "mae": float(mean_absolute_error(truth, predicted)),
        "rmse": float(mean_squared_error(truth, predicted) ** 0.5),
        "mape": float(np.nanmean(np.abs(residual) / denominator)),
        "bias": float(residual.mean()),
        "residual_std": float(residual.std()),
    }


def calibration_table(
    y_true: Sequence[int],
    probability: Sequence[float],
    bins: int = 10,
) -> pd.DataFrame:
    frame = pd.DataFrame({"target": y_true, "probability": probability})
    frame["bin"] = pd.cut(frame["probability"], bins=np.linspace(0.0, 1.0, bins + 1), include_lowest=True)
    result = (
        frame.groupby("bin", observed=True)
        .agg(observations=("target", "size"), predicted_rate=("probability", "mean"), observed_rate=("target", "mean"))
        .reset_index()
    )
    result["absolute_gap"] = (result["predicted_rate"] - result["observed_rate"]).abs()
    return result


def threshold_cost_table(
    y_true: Sequence[int],
    probability: Sequence[float],
    false_positive_cost: float,
    false_negative_cost: float,
    thresholds: Sequence[float] | None = None,
) -> pd.DataFrame:
    truth = np.asarray(y_true, dtype=int)
    probability_array = np.asarray(probability, dtype=float)
    candidates = np.asarray(thresholds if thresholds is not None else np.linspace(0.05, 0.95, 91))
    rows = []
    for threshold in candidates:
        prediction = probability_array >= threshold
        tn, fp, fn, tp = confusion_matrix(truth, prediction, labels=[0, 1]).ravel()
        cost = fp * false_positive_cost + fn * false_negative_cost
        rows.append(
            {
                "threshold": float(threshold),
                "true_negative": int(tn),
                "false_positive": int(fp),
                "false_negative": int(fn),
                "true_positive": int(tp),
                "cost": float(cost),
                "review_rate": float(prediction.mean()),
            }
        )
    return pd.DataFrame(rows)


def plot_confusion_matrix(
    y_true: Sequence[int],
    probability: Sequence[float],
    threshold: float,
    title: str,
) -> plt.Figure:
    matrix = confusion_matrix(y_true, np.asarray(probability) >= threshold, labels=[0, 1])
    figure, axis = plt.subplots(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=axis)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
    axis.set_title(title)
    return figure


def plot_numeric_distributions(frame: pd.DataFrame, columns: Sequence[str], columns_per_row: int = 3) -> plt.Figure:
    selected = list(columns)
    rows = math.ceil(len(selected) / columns_per_row)
    figure, axes = plt.subplots(rows, columns_per_row, figsize=(5 * columns_per_row, 3.5 * rows))
    axes_array = np.atleast_1d(axes).ravel()
    for axis, column in zip(axes_array, selected):
        sns.histplot(frame[column], kde=True, ax=axis)
        axis.set_title(column.replace("_", " ").title())
    for axis in axes_array[len(selected):]:
        axis.axis("off")
    figure.tight_layout()
    return figure


@dataclass
class ExperimentRecord:
    name: str
    parameters: dict[str, Any]
    metrics: dict[str, Any]
    data_fingerprint: str
    started_at: str
    elapsed_seconds: float
    notes: str = ""
    run_id: str = ""

    def __post_init__(self) -> None:
        if not self.run_id:
            self.run_id = stable_hash(
                {
                    "name": self.name,
                    "parameters": self.parameters,
                    "data_fingerprint": self.data_fingerprint,
                    "started_at": self.started_at,
                }
            )


class ExperimentTracker:
    def __init__(self, project: str):
        self.project = project
        self.records: list[ExperimentRecord] = []

    def log(
        self,
        name: str,
        parameters: Mapping[str, Any],
        metrics: Mapping[str, Any],
        data_fingerprint: str,
        elapsed_seconds: float,
        notes: str = "",
    ) -> ExperimentRecord:
        record = ExperimentRecord(
            name=name,
            parameters=dict(parameters),
            metrics=dict(metrics),
            data_fingerprint=data_fingerprint,
            started_at=utc_now(),
            elapsed_seconds=float(elapsed_seconds),
            notes=notes,
        )
        self.records.append(record)
        return record

    def to_frame(self) -> pd.DataFrame:
        rows = []
        for record in self.records:
            row = {
                "run_id": record.run_id,
                "name": record.name,
                "elapsed_seconds": record.elapsed_seconds,
                "data_fingerprint": record.data_fingerprint,
                "started_at": record.started_at,
                "notes": record.notes,
            }
            row.update({f"parameter__{key}": value for key, value in record.parameters.items()})
            row.update({f"metric__{key}": value for key, value in record.metrics.items()})
            rows.append(row)
        return pd.DataFrame(rows)

    def save(self) -> Path:
        return write_csv(self.to_frame(), "experiment_registry.csv")


TRACKER = ExperimentTracker(CONFIG.slug)


@dataclass
class DataCard:
    name: str
    purpose: str
    source: str
    rows: int
    columns: int
    target: str | None
    synthetic: bool
    known_limitations: list[str]
    prohibited_uses: list[str]
    fingerprint: str


@dataclass
class ModelCard:
    name: str
    model_type: str
    intended_use: str
    primary_metric: str
    evaluation: dict[str, Any]
    decision_threshold: float | None
    ethical_considerations: list[str]
    limitations: list[str]
    monitoring: list[str]


@dataclass
class DecisionRecord:
    decision: str
    recommendation: str
    evidence: dict[str, Any]
    assumptions: list[str]
    risks: list[str]
    next_actions: list[str]
    created_at: str = field(default_factory=utc_now)


def save_governance_pack(
    data_card: DataCard,
    decision_record: DecisionRecord,
    model_card: ModelCard | None = None,
) -> list[Path]:
    paths = [
        write_json(data_card, "data_card.json"),
        write_json(decision_record, "decision_record.json"),
    ]
    if model_card is not None:
        paths.append(write_json(model_card, "model_card.json"))
    return paths


def project_manifest(extra: Mapping[str, Any] | None = None) -> dict[str, Any]:
    manifest = {
        "project": CONFIG.title,
        "slug": CONFIG.slug,
        "generated_at": utc_now(),
        "seed": CONFIG.seed,
        "runtime": capture_versions([np, pd]),
        "artifact_directory": str(ARTIFACT_ROOT),
        "truth_first_notice": "Synthetic or educational results are not production or commercial claims.",
    }
    if extra:
        manifest.update(to_native(extra))
    return manifest


def format_currency(value: float, symbol: str = "£") -> str:
    return f"{symbol}{value:,.0f}"


def format_percentage(value: float, digits: int = 1) -> str:
    return f"{100.0 * value:.{digits}f}%"


def status_from_threshold(value: float, warning: float, critical: float, higher_is_worse: bool = True) -> str:
    if higher_is_worse:
        if value >= critical:
            return "critical"
        if value >= warning:
            return "warning"
        return "healthy"
    if value <= critical:
        return "critical"
    if value <= warning:
        return "warning"
    return "healthy"


def app_table(frame: pd.DataFrame, rows: int = 20) -> pd.DataFrame:
    clean = frame.head(rows).copy()
    for column in clean.select_dtypes(include="float").columns:
        clean[column] = clean[column].round(4)
    return clean


def smoke_test_callable(function: Callable[..., Any], arguments: Sequence[Any]) -> dict[str, Any]:
    started = time.perf_counter()
    try:
        output = function(*arguments)
        passed = output is not None
        error = None
    except Exception as exc:
        output = None
        passed = False
        error = f"{type(exc).__name__}: {exc}"
    return {
        "passed": passed,
        "elapsed_seconds": round(time.perf_counter() - started, 4),
        "output_type": type(output).__name__ if output is not None else None,
        "error": error,
    }


print(
    json.dumps(
        {
            "project": CONFIG.title,
            "seed": CONFIG.seed,
            "runtime_root": str(runtime_root()),
            "artifacts": str(ARTIFACT_ROOT),
            "colab": in_colab(),
        },
        indent=2,
    )
)


{
  "project": "ConsultAI: AI Opportunity Prioritisation Engine",
  "seed": 42,
  "runtime_root": "/home/runner/work/uni_projects/uni_projects",
  "artifacts": "/home/runner/work/uni_projects/uni_projects/consultai_application_artifacts",
  "colab": false
}


## 1. Core analytical build

Work through the data-generating process, analytical method and first decision output. The code is complete so the notebook runs end-to-end; use the markdown prompts to prepare your interview explanation.


### Task 1.1 — Data construction and contract


In [4]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json, math, random, statistics, unittest

SEED = 42
random.seed(SEED)
ARTIFACTS = Path("artifacts_python")
ARTIFACTS.mkdir(exist_ok=True)

@dataclass(frozen=True)
class AIUseCase:
    name: str
    department: str
    annual_value_gbp: float
    delivery_cost_gbp: float
    months_to_value: int
    data_readiness: int       # 1-5
    technical_feasibility: int # 1-5
    adoption_readiness: int    # 1-5
    risk: int                  # 1-5, higher is worse

    def validate(self):
        if self.annual_value_gbp <= 0 or self.delivery_cost_gbp <= 0:
            raise ValueError("Value and cost must be positive")
        if self.months_to_value not in range(1, 37):
            raise ValueError("months_to_value must be 1-36")
        for field in ("data_readiness", "technical_feasibility", "adoption_readiness", "risk"):
            if not 1 <= getattr(self, field) <= 5:
                raise ValueError(f"{field} must be 1-5")

def score(case: AIUseCase) -> float:
    case.validate()
    value_score = min(case.annual_value_gbp / case.delivery_cost_gbp, 8) / 8 * 100
    speed_score = (37 - case.months_to_value) / 36 * 100
    readiness = statistics.mean([
        case.data_readiness, case.technical_feasibility, case.adoption_readiness
    ]) / 5 * 100
    risk_score = (6 - case.risk) / 5 * 100
    return round(.35*value_score + .15*speed_score + .30*readiness + .20*risk_score, 1)

use_cases = [
    AIUseCase("Support ticket copilot", "Customer Service", 520_000, 160_000, 5, 4, 4, 4, 2),
    AIUseCase("Demand forecasting", "Operations", 740_000, 260_000, 8, 4, 4, 3, 2),
    AIUseCase("Automated CV screening", "HR", 210_000, 140_000, 6, 3, 4, 2, 5),
    AIUseCase("Invoice anomaly detection", "Finance", 430_000, 190_000, 7, 4, 4, 4, 3),
    AIUseCase("Marketing content generator", "Marketing", 180_000, 70_000, 3, 3, 5, 4, 3),
    AIUseCase("Predictive maintenance", "Facilities", 610_000, 340_000, 12, 2, 3, 2, 3),
]
ranked = sorted(((score(x), x) for x in use_cases), reverse=True, key=lambda z: z[0])
for rank, (s, x) in enumerate(ranked, 1):
    print(f"{rank}. {x.name:30s} score={s:5.1f}  cost=£{x.delivery_cost_gbp:,.0f}")

1. Support ticket copilot         score= 67.6  cost=£160,000
2. Demand forecasting             score= 62.5  cost=£260,000
3. Marketing content generator    score= 61.4  cost=£70,000
4. Invoice anomaly detection      score= 58.4  cost=£190,000
5. Predictive maintenance         score= 44.3  cost=£340,000
6. Automated CV screening         score= 41.5  cost=£140,000


### Task 1.2 — Method implementation


In [5]:
def simulate_npv(case: AIUseCase, trials=10_000, seed=SEED):
    rng = random.Random(seed + sum(map(ord, case.name)))
    outcomes = []
    for _ in range(trials):
        adoption = min(1.0, max(0.0, rng.gauss(case.adoption_readiness/5, .12)))
        delivery_multiplier = max(.75, rng.lognormvariate(0, .18))
        value_multiplier = max(.25, rng.gauss(1, .22))
        realised_value = case.annual_value_gbp * adoption * value_multiplier
        realised_cost = case.delivery_cost_gbp * delivery_multiplier
        outcomes.append(realised_value - realised_cost)
    outcomes.sort()
    return {
        "mean_npv": round(statistics.mean(outcomes), 2),
        "p10_npv": round(outcomes[int(.10*trials)], 2),
        "probability_positive": round(sum(x > 0 for x in outcomes)/trials, 4),
    }

analysis = []
for s, case in ranked:
    analysis.append({**asdict(case), "priority_score": s, **simulate_npv(case)})

BUDGET = 600_000
selected, spent = [], 0
for row in sorted(analysis, key=lambda r: (r["mean_npv"]/r["delivery_cost_gbp"]), reverse=True):
    if spent + row["delivery_cost_gbp"] <= BUDGET and row["probability_positive"] >= .70:
        selected.append(row["name"]); spent += row["delivery_cost_gbp"]

decision = {"budget_gbp": BUDGET, "selected": selected, "spend_gbp": spent, "all_use_cases": analysis}
(ARTIFACTS/"ai_opportunity_recommendation.json").write_text(json.dumps(decision, indent=2))
print(json.dumps({"selected": selected, "spend_gbp": spent}, indent=2))

{
  "selected": [
    "Support ticket copilot",
    "Marketing content generator",
    "Invoice anomaly detection"
  ],
  "spend_gbp": 420000
}


### Task 1.3 — Evaluation and first artefacts


In [6]:
class TestOpportunityEngine(unittest.TestCase):
    def test_scores_are_bounded(self):
        self.assertTrue(all(0 <= score(x) <= 100 for x in use_cases))
    def test_budget_is_respected(self):
        self.assertLessEqual(spent, BUDGET)
    def test_determinism(self):
        self.assertEqual(simulate_npv(use_cases[0], 500), simulate_npv(use_cases[0], 500))
    def test_invalid_rating_rejected(self):
        with self.assertRaises(ValueError):
            score(AIUseCase("Bad", "Test", 1, 1, 2, 8, 2, 2, 2))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestOpportunityEngine)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()

test_budget_is_respected (__main__.TestOpportunityEngine.test_budget_is_respected) ... 

ok


test_determinism (__main__.TestOpportunityEngine.test_determinism) ... 

ok


test_invalid_rating_rejected (__main__.TestOpportunityEngine.test_invalid_rating_rejected) ... 

ok


test_scores_are_bounded (__main__.TestOpportunityEngine.test_scores_are_bounded) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.003s

OK


## 2. Advanced analysis and decision engineering

Extend the compact core into an auditable, decision-grade workflow. This section adds diagnostics, uncertainty, governance artefacts and operational scenarios.


In [7]:
import itertools

analysis_frame = pd.DataFrame(analysis).sort_values("priority_score", ascending=False).reset_index(drop=True)
analysis_frame["value_cost_ratio"] = analysis_frame["annual_value_gbp"] / analysis_frame["delivery_cost_gbp"]
analysis_frame["risk_adjusted_npv"] = analysis_frame["mean_npv"] * analysis_frame["probability_positive"]
analysis_frame["downside_gap"] = analysis_frame["mean_npv"] - analysis_frame["p10_npv"]
analysis_frame["readiness_mean"] = analysis_frame[
    ["data_readiness", "technical_feasibility", "adoption_readiness"]
].mean(axis=1)

consultai_quality = QualitySuite("consultai_input_and_outputs")
consultai_quality.require_nonempty(analysis_frame)
consultai_quality.require_unique(analysis_frame, ["name"])
consultai_quality.require_range(analysis_frame["priority_score"], 0.0, 100.0, "priority_score_bounds")
consultai_quality.require_range(
    analysis_frame["probability_positive"],
    0.0,
    1.0,
    "probability_bounds",
)
consultai_quality.add(
    "budget_respected",
    spent <= BUDGET,
    spent,
    f"selected spend no greater than {BUDGET}",
)
consultai_quality.assert_all()


def evaluate_portfolio(candidate_names: Sequence[str]) -> dict[str, Any]:
    chosen = analysis_frame[analysis_frame["name"].isin(candidate_names)]
    return {
        "selected": chosen["name"].tolist(),
        "number_selected": int(len(chosen)),
        "spend_gbp": float(chosen["delivery_cost_gbp"].sum()),
        "expected_npv_gbp": float(chosen["mean_npv"].sum()),
        "p10_npv_gbp": float(chosen["p10_npv"].sum()),
        "risk_adjusted_npv_gbp": float(chosen["risk_adjusted_npv"].sum()),
        "average_priority": float(chosen["priority_score"].mean()) if len(chosen) else 0.0,
    }


def exact_portfolio_search(
    budget_gbp: float,
    minimum_probability: float = 0.70,
    maximum_projects: int | None = None,
) -> tuple[dict[str, Any], pd.DataFrame]:
    eligible = analysis_frame[analysis_frame["probability_positive"] >= minimum_probability].copy()
    rows = []
    names = eligible["name"].tolist()
    for subset_size in range(1, len(names) + 1):
        if maximum_projects is not None and subset_size > maximum_projects:
            continue
        for subset in itertools.combinations(names, subset_size):
            result = evaluate_portfolio(subset)
            if result["spend_gbp"] <= budget_gbp:
                rows.append(result)
    if not rows:
        empty = evaluate_portfolio([])
        return empty, pd.DataFrame([empty])
    candidates = pd.DataFrame(rows)
    candidates["objective"] = (
        candidates["risk_adjusted_npv_gbp"]
        + 0.10 * candidates["p10_npv_gbp"]
        + 500.0 * candidates["average_priority"]
    )
    candidates = candidates.sort_values(
        ["objective", "expected_npv_gbp"],
        ascending=False,
    ).reset_index(drop=True)
    return candidates.iloc[0].to_dict(), candidates


budget_grid = np.arange(200_000, 1_000_001, 50_000)
frontier_rows = []
for candidate_budget in budget_grid:
    recommendation, _ = exact_portfolio_search(candidate_budget)
    recommendation["budget_gbp"] = float(candidate_budget)
    recommendation["budget_utilisation"] = recommendation["spend_gbp"] / candidate_budget
    frontier_rows.append(recommendation)
portfolio_frontier = pd.DataFrame(frontier_rows)


stress_scenarios = {
    "base": {"value_multiplier": 1.00, "cost_multiplier": 1.00, "probability_shift": 0.00},
    "delivery_delay": {"value_multiplier": 0.92, "cost_multiplier": 1.20, "probability_shift": -0.05},
    "weak_adoption": {"value_multiplier": 0.72, "cost_multiplier": 1.05, "probability_shift": -0.15},
    "strong_adoption": {"value_multiplier": 1.18, "cost_multiplier": 1.02, "probability_shift": 0.08},
    "combined_downside": {"value_multiplier": 0.65, "cost_multiplier": 1.30, "probability_shift": -0.20},
}


stress_rows = []
for scenario_name, assumptions in stress_scenarios.items():
    for row in analysis_frame.to_dict("records"):
        stressed_npv = (
            row["annual_value_gbp"] * assumptions["value_multiplier"]
            - row["delivery_cost_gbp"] * assumptions["cost_multiplier"]
        )
        stressed_probability = np.clip(
            row["probability_positive"] + assumptions["probability_shift"],
            0.0,
            1.0,
        )
        stress_rows.append(
            {
                "scenario": scenario_name,
                "name": row["name"],
                "stressed_npv_gbp": stressed_npv,
                "stressed_probability_positive": stressed_probability,
                "stressed_risk_adjusted_npv": stressed_npv * stressed_probability,
            }
        )
stress_results = pd.DataFrame(stress_rows)


base_recommendation, portfolio_candidates = exact_portfolio_search(BUDGET)
recommended_names = base_recommendation["selected"]
selected_stress = stress_results[stress_results["name"].isin(recommended_names)]
portfolio_stress = (
    selected_stress.groupby("scenario", as_index=False)
    .agg(
        expected_npv_gbp=("stressed_npv_gbp", "sum"),
        risk_adjusted_npv_gbp=("stressed_risk_adjusted_npv", "sum"),
        minimum_success_probability=("stressed_probability_positive", "min"),
    )
    .sort_values("risk_adjusted_npv_gbp", ascending=False)
)


figure, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(portfolio_frontier["budget_gbp"], portfolio_frontier["risk_adjusted_npv_gbp"], marker="o")
axes[0].set_title("Budget frontier")
axes[0].set_xlabel("Budget (£)")
axes[0].set_ylabel("Risk-adjusted expected NPV (£)")
axes[1].barh(portfolio_stress["scenario"], portfolio_stress["risk_adjusted_npv_gbp"])
axes[1].set_title("Recommended portfolio under stress")
axes[1].set_xlabel("Risk-adjusted NPV (£)")
figure.tight_layout()
save_figure(figure, "consultai_decision_diagnostics.png")
plt.show()

consultai_data_card = DataCard(
    name="AI opportunity register",
    purpose="Educational prioritisation and portfolio simulation",
    source="Deterministic synthetic business cases defined in the notebook",
    rows=len(analysis_frame),
    columns=analysis_frame.shape[1],
    target=None,
    synthetic=True,
    known_limitations=[
        "Input value and cost estimates are illustrative.",
        "Dependencies between projects are not modelled.",
        "Probability distributions are assumptions rather than learned evidence.",
    ],
    prohibited_uses=[
        "Real capital allocation without validated organisational inputs",
        "Automatic approval without accountable human review",
    ],
    fingerprint=frame_fingerprint(analysis_frame),
)
consultai_decision = DecisionRecord(
    decision="Select an AI initiative portfolio within the illustrative budget",
    recommendation=f"Fund: {', '.join(recommended_names)}",
    evidence={
        "budget_gbp": BUDGET,
        "spend_gbp": base_recommendation["spend_gbp"],
        "expected_npv_gbp": base_recommendation["expected_npv_gbp"],
        "risk_adjusted_npv_gbp": base_recommendation["risk_adjusted_npv_gbp"],
    },
    assumptions=["Benefits measured over a comparable horizon", "Portfolio candidates are divisible only at project level"],
    risks=["Weak adoption", "Cost overrun", "Unmodelled programme dependencies"],
    next_actions=["Validate estimates with owners", "Run discovery pilots", "Refresh assumptions after pilot evidence"],
)
save_governance_pack(consultai_data_card, consultai_decision)
write_csv(analysis_frame, "opportunity_register.csv")
write_csv(portfolio_frontier, "budget_frontier.csv")
write_csv(portfolio_stress, "portfolio_stress_tests.csv")
write_json(project_manifest({"quality": consultai_quality.summary()}), "project_manifest.json")

display(analysis_frame)
display(portfolio_stress)


,name,department,annual_value_gbp,delivery_cost_gbp,months_to_value,data_readiness,technical_feasibility,adoption_readiness,risk,priority_score,mean_npv,p10_npv,probability_positive,value_cost_ratio,risk_adjusted_npv,downside_gap,readiness_mean
0,Support ticket copilot,Customer Service,520000,160000,5,4,4,4,2,67.6000,"252,166.6600","109,022.3300",0.9909,3.2500,"249,871.9434","143,144.3300",4.0000
1,Demand forecasting,Operations,740000,260000,8,4,4,3,2,62.5000,"177,352.4500","3,522.3400",0.9043,2.8462,"160,379.8205","173,830.1100",3.6667
2,Marketing content generator,Marketing,180000,70000,3,3,5,4,3,61.4000,"72,133.3800","21,315.0000",0.9720,2.5714,"70,113.6454","50,818.3800",4.0000
3,Invoice anomaly detection,Finance,430000,190000,7,4,4,4,3,58.4000,"148,346.9700","26,519.4500",0.9413,2.2632,"139,639.0029","121,827.5200",4.0000
4,Predictive maintenance,Facilities,610000,340000,12,2,3,2,3,44.3000,"-100,799.9200","-241,223.9200",0.1770,1.7941,"-17,841.5858","140,424.0000",2.3333
5,Automated CV screening,HR,210000,140000,6,3,4,2,5,41.5000,"-58,264.4200","-109,381.0800",0.0757,1.5000,"-4,410.6166","51,116.6600",3.0000


,scenario,expected_npv_gbp,risk_adjusted_npv_gbp,minimum_success_probability
3,strong_adoption,"1,199,400.0000","1,189,854.4000",0.9843
0,base,"950,000.0000","897,708.0000",0.9043
2,delivery_delay,"736,800.0000","659,774.8000",0.8543
4,weak_adoption,"522,300.0000","415,643.1000",0.7543
1,combined_downside,"299,000.0000","223,603.9000",0.7043


## 3. Application layer

The application wraps the analysis in guarded input validation, an auditable response object and a non-technical interface. The UI is built but not launched automatically; change `CONFIG.launch_app` to `True` if you want a live Colab share link.


In [8]:
try:
    import gradio as gr
except ImportError:
    gr = None


@dataclass
class ApplicationResponse:
    status: str
    headline: str
    recommendation: str
    metrics: dict[str, Any]
    warnings: list[str]
    audit_id: str
    created_at: str = field(default_factory=utc_now)


def application_response(
    status: str,
    headline: str,
    recommendation: str,
    metrics: Mapping[str, Any],
    warnings_list: Sequence[str] | None = None,
) -> ApplicationResponse:
    payload = {
        "status": status,
        "headline": headline,
        "recommendation": recommendation,
        "metrics": to_native(dict(metrics)),
        "warnings": list(warnings_list or []),
        "project": CONFIG.slug,
        "created_at": utc_now(),
    }
    return ApplicationResponse(
        status=status,
        headline=headline,
        recommendation=recommendation,
        metrics=dict(metrics),
        warnings=list(warnings_list or []),
        audit_id=stable_hash(payload, 20),
    )


def response_markdown(response: ApplicationResponse) -> str:
    metric_lines = []
    for key, value in response.metrics.items():
        label = str(key).replace("_", " ").title()
        if isinstance(value, float):
            rendered = f"{value:,.4f}"
        else:
            rendered = str(value)
        metric_lines.append(f"- **{label}:** {rendered}")
    warning_lines = [f"- {warning}" for warning in response.warnings]
    warnings_section = "\n".join(warning_lines) if warning_lines else "- No automated warnings."
    return (
        f"## {response.headline}\n\n"
        f"**Status:** `{response.status}`  \n"
        f"**Recommendation:** {response.recommendation}  \n"
        f"**Audit ID:** `{response.audit_id}`\n\n"
        f"### Metrics\n" + "\n".join(metric_lines) + "\n\n"
        f"### Guardrails\n{warnings_section}"
    )


def ensure_probability(value: float) -> float:
    numeric = float(value)
    if not np.isfinite(numeric):
        raise ValueError("probability must be finite")
    return float(np.clip(numeric, 0.0, 1.0))


def ensure_positive(value: float, name: str) -> float:
    numeric = float(value)
    if not np.isfinite(numeric) or numeric <= 0:
        raise ValueError(f"{name} must be a finite positive number")
    return numeric


def safe_app_call(function: Callable[..., Any], *arguments: Any) -> tuple[Any, str | None]:
    try:
        return function(*arguments), None
    except Exception as exc:
        details = f"{type(exc).__name__}: {exc}"
        return None, details


def dataframe_download(frame: pd.DataFrame, filename: str) -> str:
    target = ARTIFACT_ROOT / filename
    frame.to_csv(target, index=False)
    return str(target)


def application_health(application: Any, smoke_result: Mapping[str, Any]) -> dict[str, Any]:
    return {
        "gradio_available": gr is not None,
        "application_built": application is not None,
        "smoke_test": dict(smoke_result),
        "launch_enabled": CONFIG.launch_app,
        "checked_at": utc_now(),
    }


def maybe_launch(application: Any) -> None:
    if application is None:
        print("Application UI was not built because Gradio is unavailable.")
        print("Run the dependency cell, then rerun the application cells.")
        return
    if CONFIG.launch_app:
        application.launch(share=in_colab(), debug=False, show_error=True)
    else:
        print("Application built successfully. Edit launch_app=False to launch_app=True in the configuration cell, then rerun this cell.")


print({"gradio_available": gr is not None, "launch_app": CONFIG.launch_app})


{'gradio_available': True, 'launch_app': False}


In [9]:
def consultai_application_logic(
    name: str,
    department: str,
    annual_value_gbp: float,
    delivery_cost_gbp: float,
    months_to_value: int,
    data_readiness: int,
    technical_feasibility: int,
    adoption_readiness: int,
    risk: int,
) -> tuple[str, pd.DataFrame]:
    candidate = AIUseCase(
        name=str(name).strip() or "New AI use case",
        department=str(department).strip() or "Unspecified",
        annual_value_gbp=ensure_positive(annual_value_gbp, "annual value"),
        delivery_cost_gbp=ensure_positive(delivery_cost_gbp, "delivery cost"),
        months_to_value=int(months_to_value),
        data_readiness=int(data_readiness),
        technical_feasibility=int(technical_feasibility),
        adoption_readiness=int(adoption_readiness),
        risk=int(risk),
    )
    priority = score(candidate)
    simulation = simulate_npv(candidate, trials=5_000, seed=CONFIG.seed)
    comparison = analysis_frame[
        ["name", "department", "priority_score", "mean_npv", "p10_npv", "probability_positive"]
    ].copy()
    comparison.loc[len(comparison)] = {
        "name": candidate.name,
        "department": candidate.department,
        "priority_score": priority,
        "mean_npv": simulation["mean_npv"],
        "p10_npv": simulation["p10_npv"],
        "probability_positive": simulation["probability_positive"],
    }
    comparison = comparison.sort_values("priority_score", ascending=False).reset_index(drop=True)
    comparison.index = comparison.index + 1
    rank = int(comparison.index[comparison["name"] == candidate.name][0])
    status = "promising" if priority >= 70 and simulation["probability_positive"] >= 0.70 else "review"
    warnings_list = []
    if risk >= 4:
        warnings_list.append("High input risk requires accountable governance review.")
    if adoption_readiness <= 2:
        warnings_list.append("Low adoption readiness may dominate technical value.")
    if simulation["p10_npv"] < 0:
        warnings_list.append("The downside simulation includes negative NPV.")
    response = application_response(
        status=status,
        headline=f"{candidate.name}: priority score {priority:.1f}",
        recommendation=f"Ranks {rank} of {len(comparison)} in this illustrative opportunity set.",
        metrics={"priority_score": priority, "rank": rank, **simulation},
        warnings_list=warnings_list,
    )
    return response_markdown(response), app_table(comparison, rows=20)


consultai_smoke = smoke_test_callable(
    consultai_application_logic,
    ["Invoice review copilot", "Finance", 350_000, 120_000, 6, 4, 4, 3, 3],
)


def build_consultai_application():
    if gr is None:
        return None
    with gr.Blocks(title="ConsultAI Opportunity Engine") as application:
        gr.Markdown("# ConsultAI Opportunity Engine\nScore and stress-test an illustrative AI use case.")
        with gr.Row():
            name_input = gr.Textbox(label="Use-case name", value="Invoice review copilot")
            department_input = gr.Dropdown(
                ["Customer Service", "Operations", "Finance", "HR", "Marketing", "Other"],
                value="Finance",
                label="Department",
            )
        with gr.Row():
            value_input = gr.Number(label="Annual value (£)", value=350000)
            cost_input = gr.Number(label="Delivery cost (£)", value=120000)
            months_input = gr.Slider(1, 36, value=6, step=1, label="Months to value")
        with gr.Row():
            data_input = gr.Slider(1, 5, value=4, step=1, label="Data readiness")
            feasibility_input = gr.Slider(1, 5, value=4, step=1, label="Technical feasibility")
            adoption_input = gr.Slider(1, 5, value=3, step=1, label="Adoption readiness")
            risk_input = gr.Slider(1, 5, value=3, step=1, label="Risk")
        run_button = gr.Button("Evaluate opportunity", variant="primary")
        recommendation_output = gr.Markdown()
        comparison_output = gr.Dataframe(interactive=False)
        run_button.click(
            consultai_application_logic,
            inputs=[name_input, department_input, value_input, cost_input, months_input, data_input, feasibility_input, adoption_input, risk_input],
            outputs=[recommendation_output, comparison_output],
        )
    return application


consultai_app = build_consultai_application()
write_json(application_health(consultai_app, consultai_smoke), "application_health.json")
print(consultai_smoke)
maybe_launch(consultai_app)


{'passed': True, 'elapsed_seconds': 0.0107, 'output_type': 'tuple', 'error': None}
Application built successfully. Edit launch_app=False to launch_app=True in the configuration cell, then rerun this cell.


## 4. Automated assessment

These checks act like a university autograder and lightweight CI suite. A portfolio claim should not be used unless every required test passes.


In [10]:
import unittest


class TestConsultAIAdvanced(unittest.TestCase):
    def test_quality_suite_is_healthy(self):
        self.assertTrue(consultai_quality.summary()["healthy"])

    def test_exact_portfolio_respects_budget(self):
        recommendation, candidates = exact_portfolio_search(BUDGET)
        self.assertLessEqual(recommendation["spend_gbp"], BUDGET)
        self.assertGreaterEqual(len(candidates), 1)

    def test_portfolio_frontier_is_feasible(self):
        self.assertTrue((portfolio_frontier["spend_gbp"] <= portfolio_frontier["budget_gbp"]).all())
        self.assertTrue(portfolio_frontier["budget_utilisation"].between(0, 1).all())

    def test_stress_scenarios_are_complete(self):
        self.assertEqual(set(portfolio_stress["scenario"]), set(stress_scenarios))

    def test_application_smoke(self):
        self.assertTrue(consultai_smoke["passed"], consultai_smoke["error"])

    def test_governance_artifacts_exist(self):
        for filename in ["data_card.json", "decision_record.json", "project_manifest.json"]:
            self.assertTrue((ARTIFACT_ROOT / filename).exists(), filename)


consultai_test_result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestConsultAIAdvanced)
)
assert consultai_test_result.wasSuccessful()
print("AUTOGRADER PASS — ConsultAI")


test_application_smoke (__main__.TestConsultAIAdvanced.test_application_smoke) ... 

ok


test_exact_portfolio_respects_budget (__main__.TestConsultAIAdvanced.test_exact_portfolio_respects_budget) ... 

ok


test_governance_artifacts_exist (__main__.TestConsultAIAdvanced.test_governance_artifacts_exist) ... 

...

ok


test_stress_scenarios_are_complete (__main__.TestConsultAIAdvanced.test_stress_scenarios_are_complete) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 0.009s

OK


AUTOGRADER PASS — ConsultAI


## 5. Submission and interview defence

Before publishing:

- Run **Runtime → Restart session and run all** in Google Colab.
- Confirm the quality suite, smoke test and application build all pass.
- Keep the executed outputs in the notebook so reviewers can inspect the evidence.
- Upload this single `.ipynb` file to GitHub; no supporting project folder is required.
- State clearly that the data is generated for portfolio demonstration where applicable.
- In an interview, explain the decision, one technical trade-off, one failure mode and the next production step.

### Stretch questions

1. Which assumption has the greatest influence on the recommendation?
2. What could make the offline evaluation overstate real-world performance?
3. Which monitoring signal should trigger retraining or human review?
4. What would you change first with access to real organisational data?
